# Activation Fuction

# Activation Function

**Feed Forward Network with GELU**

In [ ]:
import torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model=512, hidden_dim=2048):
        super().__init__()

        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, d_model)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x


# ----------------------------------------

x = torch.randn(2, 10, 512)

ffn = FeedForward()

out = ffn(x)

print("Input Shape :", x.shape)
print("Output Shape:", out.shape)

Input Shape : torch.Size([2, 10, 512])
Output Shape: torch.Size([2, 10, 512])


**SwiGLU**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLU(nn.Module):

    def __init__(self,
                 d_model=512,
                 hidden_dim=1365):
        super().__init__()

        # Up Projection
        self.up_proj = nn.Linear(d_model, hidden_dim)

        # Gate Projection
        self.gate_proj = nn.Linear(d_model, hidden_dim)

        # Down Projection
        self.down_proj = nn.Linear(hidden_dim, d_model)

    def forward(self, x):

        up = self.up_proj(x)

        gate = self.gate_proj(x)

        gate = F.silu(gate)      # Swish = SiLU

        x = up * gate

        x = self.down_proj(x)

        return x


# ----------------------------------------

x = torch.randn(2,10,512)

model = SwiGLU()

out = model(x)

print("Input :",x.shape)
print("Output:",out.shape)

Input : torch.Size([2, 10, 512])
Output: torch.Size([2, 10, 512])


**Difference**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------
# Standard FFN
# ------------------------------

class FFN(nn.Module):

    def __init__(self,d_model=512,hidden=2048):
        super().__init__()

        self.fc1 = nn.Linear(d_model,hidden)
        self.fc2 = nn.Linear(hidden,d_model)

    def forward(self,x):

        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return x


# ------------------------------
# SwiGLU
# ------------------------------

class SwiGLU(nn.Module):

    def __init__(self,d_model=512,hidden=1365):
        super().__init__()

        self.up = nn.Linear(d_model,hidden)
        self.gate = nn.Linear(d_model,hidden)
        self.down = nn.Linear(hidden,d_model)

    def forward(self,x):

        up = self.up(x)

        gate = F.silu(self.gate(x))

        x = up * gate

        x = self.down(x)

        return x


# --------------------------------

x = torch.randn(4,32,512)

ffn = FFN()

swiglu = SwiGLU()

y1 = ffn(x)
y2 = swiglu(x)

print("="*60)

print("Standard FFN Output :",y1.shape)

print("SwiGLU Output       :",y2.shape)

print("="*60)

ffn_params = sum(p.numel() for p in ffn.parameters())
swiglu_params = sum(p.numel() for p in swiglu.parameters())

print("FFN Parameters    :",ffn_params)
print("SwiGLU Parameters :",swiglu_params)

Standard FFN Output : torch.Size([4, 32, 512])
SwiGLU Output       : torch.Size([4, 32, 512])
FFN Parameters    : 2099712
SwiGLU Parameters : 2099882
